In [0]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA = "default"

bronze_tables = {
    "bookings": f"{CATALOG}.{SCHEMA}.bronze_staymetrics_bookings",
    "guests": f"{CATALOG}.{SCHEMA}.bronze_staymetrics_guests",
    "rooms": f"{CATALOG}.{SCHEMA}.bronze_staymetrics_rooms",
    "rate_plans": f"{CATALOG}.{SCHEMA}.bronze_staymetrics_rate_plans",
    "room_nights": f"{CATALOG}.{SCHEMA}.bronze_staymetrics_room_nights"
}

for name, table in bronze_tables.items():
    print(name, ":", spark.catalog.tableExists(table))

print("\n===== BRONZE COUNTS =====")

for name, table in bronze_tables.items():
    print(name, ":", spark.table(table).count())

In [0]:
candidate_tables = {
    "bookings": f"{CATALOG}.{SCHEMA}.silver_staymetrics_bookings_candidate",
    "guests": f"{CATALOG}.{SCHEMA}.silver_staymetrics_guests_candidate",
    "rooms": f"{CATALOG}.{SCHEMA}.silver_staymetrics_rooms_candidate",
    "rate_plans": f"{CATALOG}.{SCHEMA}.silver_staymetrics_rate_plans_candidate",
    "room_nights": f"{CATALOG}.{SCHEMA}.silver_staymetrics_room_nights_candidate"
}

for name in bronze_tables:

    source_table = bronze_tables[name]
    target_table = candidate_tables[name]

    df = spark.table(source_table)

    # Keep every Bronze business column unchanged.
    # Add Candidate metadata for traceability.
    candidate_df = (
        df
        .withColumn("_candidate_created_at", F.current_timestamp())
        .withColumn("_candidate_schema_version",
                    F.lit("staymetrics_silver_candidate_v1"))
    )

    (
        candidate_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(
        f"{name}: {source_table} -> {target_table}"
    )

In [0]:
print("===== SILVER CANDIDATE TABLES =====")

for name, table in candidate_tables.items():

    print(f"\n===== {name.upper()} =====")
    print("Table:", table)
    print("Rows:", spark.table(table).count())

    spark.table(table).printSchema()

    display(
        spark.table(table).limit(5)
    )

In [0]:
print("===== BRONZE VS SILVER RECONCILIATION =====")

results = []

for name in bronze_tables:

    bronze_count = spark.table(bronze_tables[name]).count()
    candidate_count = spark.table(candidate_tables[name]).count()

    candidate_df = spark.table(candidate_tables[name])

    lineage_count = (
        candidate_df
        .filter(F.col("_record_hash").isNotNull())
        .count()
    )

    results.append(
        (
            name,
            bronze_count,
            candidate_count,
            lineage_count,
            "MATCH"
            if bronze_count == candidate_count
            else "CHECK"
        )
    )

display(
    spark.createDataFrame(
        results,
        [
            "entity",
            "bronze_rows",
            "candidate_rows",
            "lineage_rows",
            "status"
        ]
    )
)

In [0]:
print("===== CONTROLLED RERUN =====")

for name in bronze_tables:

    source_table = bronze_tables[name]
    target_table = candidate_tables[name]

    (
        spark.table(source_table)
        .withColumn("_candidate_created_at", F.current_timestamp())
        .withColumn(
            "_candidate_schema_version",
            F.lit("staymetrics_silver_candidate_v1")
        )
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

print("\n===== COUNTS AFTER RERUN =====")

rerun_results = []

for name in bronze_tables:

    bronze_count = spark.table(bronze_tables[name]).count()
    candidate_count = spark.table(candidate_tables[name]).count()

    rerun_results.append(
        (
            name,
            bronze_count,
            candidate_count,
            "PASS"
            if bronze_count == candidate_count
            else "CHECK"
        )
    )

display(
    spark.createDataFrame(
        rerun_results,
        [
            "entity",
            "bronze_rows",
            "candidate_rows",
            "rerun_status"
        ]
    )
)

In [0]:
print("===== WEEK 5 SILVER CANDIDATE FINAL CHECK =====")

for name, table in candidate_tables.items():

    df = spark.table(table)

    required_lineage = [
        "_source_file_name",
        "_source_file_path",
        "_ingestion_run_id",
        "_record_hash"
    ]

    missing = [
        c for c in required_lineage
        if c not in df.columns
    ]

    print(
        name,
        "| rows =", df.count(),
        "| lineage =",
        "PASS" if not missing else "CHECK"
    )

print("\nWeek 5 Candidate tables created and validated.")